In [7]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Cargar datos
df = pd.read_csv(r'C:\Users\Cami\Downloads\Mapas2\Accidentes.csv', sep=';')
print(df.columns.tolist())

['Clase de accidente', 'Tipo de accidente', 'Año', 'Hora', 'Hora recodificada', 'Provincia ', 'Cantón', 'Distrito', 'Ruta', 'Kilómetro', 'Rural o urbano', 'Calzada vertical', 'Calzada horizontal', 'Tipo de calzada', 'Tipo de circulación', 'Estado del tiempo', 'Estado de la calzada', 'Región Mideplan', 'Tipo ruta', 'Día ', 'Mes ']


In [8]:

# 1. Agrupamos por Provincia para contar accidentes
df_provincia = df.groupby('Provincia ').size().reset_index(name='Total Accidentes')

# Nota: Para que este mapa funcione, necesitas un archivo .json con los límites de CR.
# Pero puedes probar la lógica de visualización así:
fig = px.choropleth(df_provincia,
                    locations='Provincia ',
                    color='Total Accidentes',
                    locationmode='country names', # Esto es genérico, lo ideal es el GeoJSON
                    hover_name='Provincia ',
                    title='Distribución de Accidentes por Provincia (2018-2024)',
                    color_continuous_scale='Reds')
fig.show(renderer="browser")

C:\Users\Cami\AppData\Local\Temp\ipykernel_10300\466060064.py:6: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(df_provincia,


In [9]:
import pandas as pd
import plotly.express as px

df = pd.read_csv(r'C:\Users\Cami\Downloads\Mapas2\Accidentes.csv', sep=';')

# Agrupamos para que el gráfico no sea pesado
df_resumen = df.groupby(['Provincia ', 'Cantón', 'Clase de accidente']).size().reset_index(name='Total')

fig = px.treemap(df_resumen,
                 path=['Provincia ', 'Cantón'],
                 values='Total',
                 color='Clase de accidente',
                 color_discrete_map={'Solo heridos leves': '#2980b9', 'Con muertos o graves': '#c0392b'},
                 title='Jerarquía de Accidentes por Provincia y Cantón')

fig.show(renderer="browser")

In [10]:
# Coordenadas aproximadas de las cabeceras de provincia
coords = {
    'San José': [9.9333, -84.0833], 'Alajuela': [10.0167, -84.2167],
    'Cartago': [9.8644, -83.9194], 'Heredia': [10.0024, -84.1165],
    'Guanacaste': [10.6333, -85.4333], 'Puntarenas': [9.9763, -84.8389],
    'Limón': [9.9913, -83.0358]
}

df_mapa = df.groupby(['Provincia ', 'Clase de accidente']).size().reset_index(name='Total')
df_mapa['lat'] = df_mapa['Provincia '].map(lambda x: coords.get(x.strip(), [0,0])[0])
df_mapa['lon'] = df_mapa['Provincia '].map(lambda x: coords.get(x.strip(), [0,0])[1])

fig_map = px.scatter_mapbox(df_mapa,
                            lat="lat", lon="lon",
                            size="Total",
                            color="Clase de accidente",
                            hover_name="Provincia ",
                            zoom=6.5,
                            mapbox_style="open-street-map",
                            title='Concentración de Accidentes por Provincia')

fig.show(renderer="browser")

C:\Users\Cami\AppData\Local\Temp\ipykernel_10300\3006998066.py:13: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_map = px.scatter_mapbox(df_mapa,


In [11]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Configuración global para abrir en navegador (soluciona el error de conexión)
pio.renderers.default = "browser"

# 1. CARGA Y LIMPIEZA (Se hace solo una vez)
path = r'C:\Users\Cami\Downloads\Mapas2\Accidentes.csv'
df = pd.read_csv(path, sep=';')

# Limpiamos nombres de columnas (quita espacios molestos como 'Provincia ')
df.columns = df.columns.str.strip()
# Limpiamos el contenido de las celdas (por si hay "San José " con espacio)
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Agrupamos una sola vez para este gráfico
df_tree = df.groupby(['Provincia', 'Cantón', 'Clase de accidente']).size().reset_index(name='Total')

fig_tree = px.treemap(df_tree,
                      path=['Provincia', 'Cantón', 'Clase de accidente'],
                      values='Total',
                      color='Clase de accidente',
                      color_discrete_map={'Solo heridos leves': '#2980b9', 'Con muertos o graves': '#c0392b'},
                      title='Análisis Jerárquico: Provincias y Cantones')
fig_tree.show()

coords = {
    'San José': [9.9333, -84.0833], 'Alajuela': [10.0167, -84.2167],
    'Cartago': [9.8644, -83.9194], 'Heredia': [10.0024, -84.1165],
    'Guanacaste': [10.6333, -85.4333], 'Puntarenas': [9.9763, -84.8389],
    'Limón': [9.9913, -83.0358]
}

# Agrupamos por provincia y gravedad
df_mapa = df.groupby(['Provincia', 'Clase de accidente']).size().reset_index(name='Total')

# Mapeo de coordenadas rápido
df_mapa['lat'] = df_mapa['Provincia'].map(lambda x: coords.get(x, [0, 0])[0])
df_mapa['lon'] = df_mapa['Provincia'].map(lambda x: coords.get(x, [0, 0])[1])

fig_map = px.scatter_mapbox(df_mapa,
                            lat="lat", lon="lon",
                            size="Total",
                            color="Clase de accidente",
                            hover_name="Provincia",
                            hover_data={"Total": True, "lat": False, "lon": False},
                            zoom=6.8,
                            mapbox_style="carto-positron",  # Estilo más limpio
                            title='Concentración de Accidentes por Provincia (2018-2024)')

fig_map.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig_map.show()

# Este gráfico permite comparar la peligrosidad relativa
fig_bar = px.bar(df_mapa, x="Provincia", y="Total", color="Clase de accidente",
                 title="Comparativa de Gravedad por Provincia",
                 barmode="group",
                 text_auto='.2s')
fig_bar.show()

C:\Users\Cami\AppData\Local\Temp\ipykernel_10300\539551009.py:42: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_map = px.scatter_mapbox(df_mapa,


In [12]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "browser"

# 1. Carga y Limpieza profunda
path = r'C:\Users\Cami\Downloads\Mapas2\Accidentes.csv'
df = pd.read_csv(path, sep=';')
df.columns = df.columns.str.strip()
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# 2. Coordenadas base por provincia
coords = {
    'San José': [9.9333, -84.0833], 'Alajuela': [10.0167, -84.2167],
    'Cartago': [9.8644, -83.9194], 'Heredia': [10.0024, -84.1165],
    'Guanacaste': [10.6333, -85.4333], 'Puntarenas': [9.9763, -84.8389],
    'Limón': [9.9913, -83.0358]
}

# 3. Agrupación por Año, Provincia y Clase (Para el Slider)
df_mapa = df.groupby(['Año', 'Provincia', 'Clase de accidente']).size().reset_index(name='Total')

# 4. Aplicar coordenadas y Jittering (El truco para que no se traslapen)
def get_coords_with_jitter(row):
    lat, lon = coords.get(row['Provincia'], [0, 0])
    # Si es grave, lo movemos un poquito a la derecha/arriba
    if row['Clase de accidente'] == 'Con muertos o graves':
        return lat + 0.04, lon + 0.04
    # Si es leve, lo movemos un poquito a la izquierda/abajo
    else:
        return lat - 0.04, lon - 0.04

df_mapa[['lat', 'lon']] = df_mapa.apply(
    lambda row: pd.Series(get_coords_with_jitter(row)), axis=1
)

# 5. Creación del Mapa Animado y Mejorado
fig_map = px.scatter_mapbox(
    df_mapa,
    lat="lat",
    lon="lon",
    size="Total",
    color="Clase de accidente",
    animation_frame="Año",  # ESTO CREA EL SLIDER
    hover_name="Provincia",
    hover_data={"Total": True, "lat": False, "lon": False, "Año": False},
    size_max=40,            # Ajusta el tamaño de las burbujas
    zoom=6.8,
    color_discrete_map={'Solo heridos leves': '#2980b9', 'Con muertos o graves': '#c0392b'},
    mapbox_style="carto-positron",
    title='Evolución de Accidentes en Costa Rica (2018-2024)'
)

# Ajustes estéticos finales
fig_map.update_layout(
    margin={"r":0,"t":50,"l":0,"b":0},
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig_map.show()

C:\Users\Cami\AppData\Local\Temp\ipykernel_10300\3696231725.py:39: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_map = px.scatter_mapbox(
